# NEXUS § 133 — Ask the field: every open item of §§ 125–132

> *"anything OPEN, just ask the field."* — Dean, 17 Oct

Each gap is named by what must be there. The forward route is subtracted, and the remainder is bound to its owner. Seals were written before any data was linked:

- `PREDICTION_ask_field.txt`: `f97de11f49e2642837197d393d2b055c95a4aeb1b2dfd1e1fdd69ab6caf154bb` (G1–G4);
- `PREDICTION_core_stage2.txt`: `77a752092108de26220df059ebd5778a5029937d904acaa1da4215996a0be89b` (seven new layers, predicted from the returns alone).

Both seal texts are embedded and re-hashed in the last cell.

**Result in one line.** The zero routes of every lane machine are the Galois-stable multiplicative relations among its wheel returns $\beta=\sqrt p-\zeta$. Binary is the only base in which two prime-free returns, phase and scale, meet. That meeting is BBP's zero route, and the register-chain proof shows it is the only one at every layer.

In [18]:
RUN_SLOW = True   # lane PSLQ at p=3 n=24, p=2 n=28, p=2 n=36 (long); False for a quick pass

In [19]:
import math, time, hashlib
import sympy as sp
from mpmath import mp, mpf, mpc, exp, pi, log, sqrt, arg, pslq, fabs, log10
x = sp.symbols('x')
checks = {}
def check(name, ok):
    checks[name] = bool(ok); print(("PASS " if ok else "FAIL ") + name, flush=True)

def reldim(vals, maxc=10**12):
    # integer relation space by PSLQ with elimination; returns (relations, surviving basis indices)
    idx, rels = list(range(len(vals))), []
    while len(idx) >= 2:
        r = pslq([vals[i] for i in idx], maxcoeff=maxc, maxsteps=10**7)
        if r is None: break
        assert fabs(sum(a * vals[i] for a, i in zip(r, idx))) < mpf(10) ** (-(mp.dps * 3 // 4))
        rels.append({i: a for i, a in zip(idx, r) if a}); idx.remove(max(i for i, a in zip(idx, r) if a))
    return rels, idx

def disc(p): return 8 if p == 2 else (p if p % 4 == 1 else 4 * p)        # discriminant of Q(sqrt p)

def index_maps(p, n):
    # Galois action on the returns: beta_m -> +-beta_(a m + [sqrt p -> -sqrt p] n/2)
    D = disc(p); N = math.lcm(n, D); maps = []
    for a in range(1, N):
        if math.gcd(a, N) != 1: continue
        flip = sp.kronecker_symbol(D, a) == -1
        maps.append([(a * m + (n // 2 if flip else 0)) % n for m in range(n)])
    return maps

def relation_space(p, n, dps):
    # Q-relations among L_m = -log(1 - zeta_n^m / sqrt p), as integer vectors on m = 0..n-1
    mp.dps = dps
    al = [1 - exp(2j * pi * m / n) / sqrt(p) for m in range(n)]
    re_ = [log(abs(al[m])) for m in range(n // 2 + 1)]          # moduli: a_m = (L_m + L_{n-m})/2
    im_ = [arg(al[m]) for m in range(1, n // 2)]                # arguments: b_m = (L_m - L_{n-m})/2i
    rr, _ = reldim(re_); ri, _ = reldim(im_)
    rows = []
    for r in rr:
        e = [0] * n
        for m, c in r.items():
            if m in (0, n // 2): e[m] += 2 * c
            else: e[m] += c; e[n - m] += c
        rows.append(e)
    for r in ri:
        e = [0] * n
        for k, c in r.items(): e[k + 1] += c; e[n - k - 1] -= c
        rows.append(e)
    return rows, rr, ri

def galois_core_dim(p, n, rows):
    if not rows: return 0
    R = sp.Matrix(rows); ann = R.nullspace()
    if not ann: return n
    A = sp.Matrix.hstack(*ann).T; stack = []
    for mp_ in index_maps(p, n):
        P = sp.zeros(n, n)
        for m in range(n): P[mp_[m], m] = 1
        stack.append(A * P.T)
    return n - sp.Matrix.vstack(*stack).rank()

def lane_run(p, n, dps, members=True):
    mp.dps = dps; b = mpf(p) ** (n // 2); terms = int(dps / float(log10(b))) + 12
    L = [sum(1 / (b ** k * (n * k + j)) for k in range(terms)) for j in range(1, n + 1)]
    rels, basis = reldim(L)
    mem = {}
    if members:
        for nm, c in [("pi", pi), ("pi*sqrt3", pi * sqrt(3)), ("pi*sqrt2", pi * sqrt(2))]:
            mem[nm] = pslq([c] + [L[i] for i in basis], maxcoeff=10**12, maxsteps=10**7) is not None
    return len(rels), rels, mem
print("library ready")

library ready


## 1 · The join (§§ 117, 126, 128): two prime-free returns meet only in binary

Take base $p^{n/2}$ with wheels $w_m=\zeta_n^m/\sqrt p$. The returns are $\beta_m=\sqrt p-\zeta_n^m$, and $\prod_m\beta_m=p^{n/2}-1$: the base-$p$ register ring. A return costs no prime exactly when it is a unit. Two kinds can be prime-free:

* **Scale return** ($\zeta=\pm1$): $N(\sqrt p-1)=1-p$. It is a unit **only at $p=2$**. For every odd $p$ the scale return pays $p-1$, which is even, so binary sits inside every odd stack.
* **Phase return** ($\beta$ a root of unity): this needs $|\sqrt p-\zeta|=1$. Two unit circles whose centres are $\sqrt p$ apart meet only when $\sqrt p\le2$, so $\cos\theta=\sqrt p/2$ and $p\in\{2,3\}$. Equivalently $1-w=\bar w$, which puts the wheel at $\operatorname{Re}w=\tfrac12$, in the middle. The cases are $\sqrt2-\zeta_8=\zeta_8^{-1}$ (angle $\pi/4$) and $\sqrt3-\zeta_{12}=\zeta_{12}^{-1}$ (angle $\pi/6$, § 126's exception).

So $p=2$ is the only prime with both kinds prime-free. That pair is the phase/scale collision of § 131, BBP's zero route. § 126's quadratic boundary ($p=2,3$ only) is the two-circle condition, and § 117's "the stack holds only the prime 2" is $N(\sqrt p-1)=1-p$.

In [20]:
check("scale return: |N(sqrt p - 1)| = p - 1 for all primes p < 60 (a unit only at p = 2)",
      all(abs(sp.expand((sp.sqrt(p) - 1) * (-sp.sqrt(p) - 1))) == p - 1 for p in sp.primerange(2, 60)))
check("phase return needs sqrt(p) <= 2: primes {2, 3} only", [p for p in sp.primerange(2, 10**4) if p <= 4] == [2, 3])
for p, n in [(2, 8), (3, 12)]:
    z = sp.exp(2 * sp.pi * sp.I / n); w = z / sp.sqrt(p)
    check(f"p={p}: sqrt{p} - zeta_{n} = zeta_{n}^-1;  w = {sp.nsimplify(sp.expand_complex(w))} has Re w = 1/2 and 1 - w = conj(w)",
          sp.simplify(sp.expand_complex(sp.sqrt(p) - z - z**-1)) == 0 and
          sp.simplify(sp.re(sp.expand_complex(w)) - sp.Rational(1, 2)) == 0 and
          sp.simplify(sp.expand_complex(1 - w - sp.conjugate(w))) == 0)
mp.dps = 80
for p, n in [(2, 8), (2, 16), (3, 12), (3, 24), (5, 8), (5, 12)]:
    pr = mpc(1)
    for m in range(n): pr *= sqrt(p) - exp(2j * pi * m / n)
    check(f"prod (sqrt{p} - zeta_{n}^m) = {p}^{n//2} - 1 = {dict(sp.factorint(p**(n//2)-1))}", fabs(pr - (p**(n//2) - 1)) < mpf(10)**-60)

PASS scale return: |N(sqrt p - 1)| = p - 1 for all primes p < 60 (a unit only at p = 2)
PASS phase return needs sqrt(p) <= 2: primes {2, 3} only
PASS p=2: sqrt2 - zeta_8 = zeta_8^-1;  w = 1/2 + I/2 has Re w = 1/2 and 1 - w = conj(w)
PASS p=3: sqrt3 - zeta_12 = zeta_12^-1;  w = 1/2 + sqrt(3)*I/6 has Re w = 1/2 and 1 - w = conj(w)
PASS prod (sqrt2 - zeta_8^m) = 2^4 - 1 = {3: 1, 5: 1}
PASS prod (sqrt2 - zeta_16^m) = 2^8 - 1 = {3: 1, 5: 1, 17: 1}
PASS prod (sqrt3 - zeta_12^m) = 3^6 - 1 = {2: 3, 7: 1, 13: 1}
PASS prod (sqrt3 - zeta_24^m) = 3^12 - 1 = {2: 4, 5: 1, 7: 1, 13: 1, 73: 1}
PASS prod (sqrt5 - zeta_8^m) = 5^4 - 1 = {2: 4, 3: 1, 13: 1}
PASS prod (sqrt5 - zeta_12^m) = 5^6 - 1 = {2: 3, 3: 2, 7: 1, 31: 1}


## 2 · The law: zero routes are the Galois core of the returns' relations

**First reading, sealed as G2: DEAD END off the chain.** The guess was that the zero routes are all Q-relations among $L_m=-\log(1-w_m)$. On the chain it lands ($n=8,16,32$). Off the chain it over-counts: $2$ against a measured $1$ at $n=12$, and $7$ against $5$ at $n=24$.

**Routed to its owner.** A rational machine is a zero route iff its orbit tuple $(P_c(\gamma_m))_m$ lies in $R\otimes\overline{\mathbb Q}$; that equivalence uses Baker. That tuple is fixed by the twisted Galois action, so it lies in every conjugate $\pi_\sigma(R)$. Galois descent then gives

$$\boxed{\;Z_p(n)=\dim R_0,\qquad R_0=\bigcap_\sigma \pi_\sigma(R).\;}$$

A relation that holds at one embedding but not at its conjugates is visible to the returns but to no rational machine. At $n=12$ that relation is $2b_1-b_3-4b_5=0$: under $\sigma_7$ it becomes $6(b_1-b_5)\ne0$.

In [21]:
measured = {8: 1, 12: 1, 16: 1, 24: 5, 32: 1}          # lane PSLQ, section 131 (|coeff| <= 1e12)
for n, z in measured.items():
    t0 = time.time(); rows, rr, ri = relation_space(2, n, 300 if n <= 16 else 420)
    dR, dR0 = sp.Matrix(rows).rank(), galois_core_dim(2, n, rows)
    print(f"   n={n:2d}: relations among returns {dR} (modulus {len(rr)}, argument {len(ri)})   Galois core {dR0}   measured Z {z}   [{time.time()-t0:.0f}s]")
    if n == 12: print("        argument relation (indices m-1):", ri)
    check(f"route back n={n}: Galois core {dR0} = measured Z({n}) = {z}", dR0 == z)
    if n in (12, 24): check(f"naive law (all relations) misses at n={n}: {dR} != {z}  [DEAD END, routed to the core]", dR != z)

   n= 8: relations among returns 1 (modulus 1, argument 0)   Galois core 1   measured Z 1   [0s]
PASS route back n=8: Galois core 1 = measured Z(8) = 1
   n=12: relations among returns 2 (modulus 1, argument 1)   Galois core 1   measured Z 1   [0s]
        argument relation (indices m-1): [{0: 2, 2: -1, 4: -4}]
PASS route back n=12: Galois core 1 = measured Z(12) = 1
PASS naive law (all relations) misses at n=12: 2 != 1  [DEAD END, routed to the core]
   n=16: relations among returns 1 (modulus 1, argument 0)   Galois core 1   measured Z 1   [1s]
PASS route back n=16: Galois core 1 = measured Z(16) = 1
   n=24: relations among returns 7 (modulus 3, argument 4)   Galois core 5   measured Z 5   [2s]
PASS route back n=24: Galois core 5 = measured Z(24) = 5
PASS naive law (all relations) misses at n=24: 7 != 5  [DEAD END, routed to the core]
   n=32: relations among returns 1 (modulus 1, argument 0)   Galois core 1   measured Z 1   [20s]
PASS route back n=32: Galois core 1 = measured Z(32)

## 3 · Landing: seven new layers predicted from the returns alone, then measured

Sealed in `77a75209…`, before any lane value was computed:

| layer | base | predicted $Z$ | why (owner) |
|---|---|--:|---|
| $p=2,\ n=20$ | $2^{10}$ | 0 | no $\zeta_8$ phase return; $1023=3\cdot11\cdot31$ squarefree |
| $p=2,\ n=28$ | $2^{14}$ | 0 | no phase return; $3\cdot43\cdot127$ |
| $p=2,\ n=36$ | $2^{18}$ | 1 | $3^3\mid2^{18}-1$: the repeated prime collides (= $Z(12)$ pulled back) |
| $p=3,\ n=12$ | $3^{6}$ | 2 | phase return $\zeta_{12}^{-1}$; $728=2^3\cdot7\cdot13$ |
| $p=3,\ n=24$ | $3^{12}$ | 2 | |
| $p=5,\ n=8$ | $5^4$ | 0 | no phase return for $p\ge5$ |
| $p=5,\ n=12$ | $5^6$ | 0 | |

Membership was sealed with them. For $p=3$, $\pi\sqrt3$ is in the lane span and $\pi$ is not, because the phase return pairs with $\sqrt{-3}$ coefficients. For $p=2$ at $n=20,28$, $\pi$ is not in the span.

In [22]:
pred = {(2, 20): 0, (2, 28): 0, (2, 36): 1, (3, 12): 2, (3, 24): 2, (5, 8): 0, (5, 12): 0}
for (p, n), z in pred.items():
    rows, _, _ = relation_space(p, n, 300 if n <= 16 else 420); g = galois_core_dim(p, n, rows)
    check(f"field side: Galois core at p={p}, n={n} = {g} (sealed {z})", g == z)
fast = [(3, 12, 400), (5, 8, 300), (5, 12, 400), (2, 20, 600)]
slow = [(3, 24, 700), (2, 28, 800), (2, 36, 900)]
for p, n, dps in fast + (slow if RUN_SLOW else []):
    t0 = time.time(); zn, rels, mem = lane_run(p, n, dps, members=(n != 36))
    print(f"   lanes p={p} n={n:2d}: Z = {zn}   membership {mem}   [{time.time()-t0:.0f}s]")
    check(f"LANDING p={p} n={n}: lane Z = {zn} = predicted {pred[(p, n)]}", zn == pred[(p, n)])
    if p == 3: check(f"p=3 n={n}: pi*sqrt3 in span, pi not (sealed)", mem["pi*sqrt3"] and not mem["pi"])
    if p == 2 and n in (20, 28): check(f"p=2 n={n}: pi not in span (sealed)", not mem["pi"])

PASS field side: Galois core at p=2, n=20 = 0 (sealed 0)
PASS field side: Galois core at p=2, n=28 = 0 (sealed 0)
PASS field side: Galois core at p=2, n=36 = 1 (sealed 1)
PASS field side: Galois core at p=3, n=12 = 2 (sealed 2)
PASS field side: Galois core at p=3, n=24 = 2 (sealed 2)
PASS field side: Galois core at p=5, n=8 = 0 (sealed 0)
PASS field side: Galois core at p=5, n=12 = 0 (sealed 0)
   lanes p=3 n=12: Z = 2   membership {'pi': False, 'pi*sqrt3': True, 'pi*sqrt2': False}   [6s]
PASS LANDING p=3 n=12: lane Z = 2 = predicted 2
PASS p=3 n=12: pi*sqrt3 in span, pi not (sealed)
   lanes p=5 n= 8: Z = 0   membership {'pi': False, 'pi*sqrt3': False, 'pi*sqrt2': False}   [2s]
PASS LANDING p=5 n=8: lane Z = 0 = predicted 0
   lanes p=5 n=12: Z = 0   membership {'pi': False, 'pi*sqrt3': False, 'pi*sqrt2': False}   [15s]
PASS LANDING p=5 n=12: lane Z = 0 = predicted 0
   lanes p=2 n=20: Z = 0   membership {'pi': False, 'pi*sqrt3': False, 'pi*sqrt2': False}   [211s]
PASS LANDING p=2 n=2

## 4 · The register chain, proved: $Z(2^e)=1$ at every layer, and the route is $\pi^*Z(8)$

On $n=2^e$ the orbits and their return products are:

* scale, product 1 (units $\sqrt2\mp1$);
* phase, product 1 ($\zeta_8^{\mp1}$);
* $3$, $5$;
* one orbit per Fermat number $F_2,\dots,F_{e-2}$.

The products are pairwise coprime (Fermat), and they multiply to $2^{2^{e-1}}-1$.

$\operatorname{ord}_{2^e}(3)=\operatorname{ord}_{2^e}(5)=2^{e-2}$, so 3 and 5 each have exactly two primes in $\mathbb Q(\zeta_{2^e})$, one per orbit member. Every prime factor of $F_k$ is $\equiv1\pmod{2^{k+2}}$ (Lucas, STANDS), so it splits completely, and the conjugate members of its orbit own pairwise distinct primes.

Take any multiplicative relation among the returns. Its valuation at each of those primes is a single exponent, so every exponent on a prime-orbit member vanishes (unique factorization). By Baker, the same holds for log relations. What remains are the four prime-free returns. Their moduli satisfy exactly one relation, $a_0+a_{n/2}=2a_{n/8}$, which says the scale pair and the phase pair each cost one bit, $\tfrac12$. Their arguments ($\pm\pi/4$ and $0$) satisfy none. That relation is Galois-stable, so $Z(2^e)=1$. $\pi^*Z(8)$ is a nonzero zero route, so it spans. $\square$

In [23]:
def orbit_count_ok(e):
    n = 2 ** e; N = math.lcm(n, 8); seen, orbs = set(), []
    for m in range(n):
        if m in seen: continue
        o = {(a * m + (n // 2 if a % 8 in (3, 5) else 0)) % n for a in range(1, N) if math.gcd(a, N) == 1}
        seen |= o; orbs.append(sorted(o))
    top = [o for o in orbs if 1 in o][0]
    return len(orbs) == 4 + max(0, e - 3) and (e < 4 or top == list(range(1, n, 2)))
check("orbits on n = 2^e, e = 3..12: scale, phase, 3, 5 and one per order 2^f, f = 4..e", all(orbit_count_ok(e) for e in range(3, 13)))
check("ord_(2^e)(3) = ord_(2^e)(5) = 2^(e-2), e = 3..14", all(sp.n_order(3, 2**e) == 2**(e-2) == sp.n_order(5, 2**e) for e in range(3, 15)))
check("Lucas: every prime factor of F_2..F_6 is 1 mod 2^(k+2)",
      all((q - 1) % 2 ** (k + 2) == 0 for k in range(2, 7) for q in sp.factorint(2 ** (2 ** k) + 1)))
check("Fermat numbers F_0..F_8 pairwise coprime", all(math.gcd(2**(2**i)+1, 2**(2**j)+1) == 1 for i in range(9) for j in range(i)))
for e in (3, 4, 5):
    rows, rr, ri = relation_space(2, 2 ** e, 300 if e < 5 else 420)
    rel = rr[0] if rr else {}
    check(f"n={2**e}: the only relation among the returns is a_0 + a_(n/2) = 2 a_(n/8): {rel}; arguments free",
          len(rr) == 1 and not ri and rel in ({0: 1, 2**e // 8: -2, 2**e // 2: 1}, {0: -1, 2**e // 8: 2, 2**e // 2: -1}))

PASS orbits on n = 2^e, e = 3..12: scale, phase, 3, 5 and one per order 2^f, f = 4..e
PASS ord_(2^e)(3) = ord_(2^e)(5) = 2^(e-2), e = 3..14
PASS Lucas: every prime factor of F_2..F_6 is 1 mod 2^(k+2)
PASS Fermat numbers F_0..F_8 pairwise coprime
PASS n=8: the only relation among the returns is a_0 + a_(n/2) = 2 a_(n/8): {0: 1, 1: -2, 4: 1}; arguments free
PASS n=16: the only relation among the returns is a_0 + a_(n/2) = 2 a_(n/8): {0: 1, 2: -2, 8: 1}; arguments free
PASS n=32: the only relation among the returns is a_0 + a_(n/2) = 2 a_(n/8): {0: 1, 4: -2, 16: 1}; arguments free


## 5 · The bit bridge (§ 125's PULL, flagged by GPT)

§ 125 read $|w|^2=\tfrac12$ as "one bit per tick", and GPT asked for the map between the two. It is this. Multiplication by $w=a+bi$ on $\mathbb R^2$ is the matrix $\begin{psmallmatrix}a&-b\\ b&a\end{psmallmatrix}$ with determinant $|w|^2$: **an area Jacobian, not a probability**.

The inverse wheel $1/s=1-i$ acts on the torus $\mathbb C/\mathbb Z[i]$ as an expanding endomorphism with eigenvalues $1\pm i$ and degree $N(1-i)=2$. Its index on $k$ ticks is $|\mathbb Z[i]/(1-i)^k|=2^k$, which is one binary digit per tick. Its entropy is $\log|\det|=\log2$, exactly one bit (expanding toral endomorphisms, STANDS).

Identifying this with § 123's measured ring leak, also one degree-2 merge per tick, stays a reading (PULL).

In [24]:
from sympy.matrices.normalforms import smith_normal_form
Ms = sp.Matrix([[sp.Rational(1, 2), -sp.Rational(1, 2)], [sp.Rational(1, 2), sp.Rational(1, 2)]])
M = sp.Matrix([[1, 1], [-1, 1]])
check(f"det(multiplication by s) = {Ms.det()} = |s|^2", Ms.det() == sp.Rational(1, 2))
check("1 - i = 1/s: matrix [[1,1],[-1,1]], det 2, eigenvalues 1 +- i (|.| = sqrt2 > 1, expanding)",
      sp.simplify(Ms.inv() - M) == sp.zeros(2) and M.det() == 2 and all(sp.Abs(e) == sp.sqrt(2) for e in M.eigenvals()))
check("|Z[i]/(1-i)^k| = 2^k, k = 1..12 (Smith normal form)",
      all(abs(sp.prod(smith_normal_form(M ** k, domain=sp.ZZ).diagonal())) == 2 ** k for k in range(1, 13)))

PASS det(multiplication by s) = 1/2 = |s|^2
PASS 1 - i = 1/s: matrix [[1,1],[-1,1]], det 2, eigenvalues 1 +- i (|.| = sqrt2 > 1, expanding)
PASS |Z[i]/(1-i)^k| = 2^k, k = 1..12 (Smith normal form)


## 6 · Ledger

| open item | answer from the field | status |
|---|---|---|
| § 126 / § 128: one theorem joining § 117's stack and the phase boundary | returns multiply to $p^{n/2}-1$; the scale return is prime-free iff $p=2$; a phase return exists iff $\sqrt p\le2$ ($p=2,3$; $\operatorname{Re}w=\tfrac12$, $1-w=\bar w$); only binary has both | RUNS (exact); bounds STAND |
| off-chain $Z(n)$ (measured, unpredicted in § 131) | $Z_p(n)=\dim$ Galois core of the returns' relations | route back 5/5; **landed** on the new layers as sealed |
| the naive law (all relations) | over-counts off the chain | DEAD END, owner: non-stable relations |
| § 131.7 / § 132: $Z(2^e)=1$ for all $e$, route $=\pi^*Z(8)$ | proof in § 4 (Fermat coprimality, Lucas, valuations, Baker) | proof written; lemmas RUN to $e=12$–$14$; inputs STAND |
| § 125: one bit $\Leftrightarrow|w|^2=\tfrac12$ | determinant of $z\mapsto wz$; inverse wheel $1-i$ has degree 2, entropy $\log2$ | RUNS (exact) + STANDS; tie to § 123's leak PULL |
| § 127: the twisted two-sheet lift | both directions are native (plain $n$, Johnson $2n$); the reader $D$ is blind (§ 131.8) | closed: two routes, not a missing law |
| base 3 | the Eisenstein phase wheel reads $\pi\sqrt3$, never $\pi$ | RUNS (sealed) |

In [25]:
for name, h, txt in [("ask_field", "f97de11f49e2642837197d393d2b055c95a4aeb1b2dfd1e1fdd69ab6caf154bb", "NEXUS - ask the field on every open item, 17 Oct 2026 - sealed before any run.\nG1 join (sections 117, 126, 128). Base p^(n/2), wheels w = zeta_n^m/sqrt(p), returns beta = sqrt(p) - zeta_n^m,\n   prod_m beta_m = p^(n/2) - 1 (base-p register ring). Scale return sqrt(p) -+ 1 has norm p - 1: prime-free iff p = 2.\n   Phase return (beta a root of unity) iff |sqrt(p) - zeta| = 1 iff cos(theta) = sqrt(p)/2 iff p in {2,3}:\n   sqrt2 - zeta_8 = zeta_8^-1, sqrt3 - zeta_12 = zeta_12^-1; equivalently 1 - w = conj(w), Re w = 1/2.\n   p = 2 is the only prime with both returns prime-free -> the phase/scale collision (BBP's zero route) is binary-only.\n   p >= 5: no phase return (circles |z|=1, |z - sqrt p| = 1 do not meet).\nG2 law. Zero routes are the rational relations among the returns' logs, descended through the lane map:\n   Z(n) = [n/2 + 1 - dRe(n)] + [n/2 - 1 - dIm(n)], dRe = dim_Q span{log|1 - zeta_n^m/sqrt2| : 0<=m<=n/2},\n   dIm = dim_Q span{arg(1 - zeta_n^m/sqrt2) : 1<=m<n/2}. Reproduces measured Z = 1,1,1,5,1 at n = 8,12,16,24,32.\n   At n = 8: V.Z = 64 e, e = (L0 + L4) - (L1 + L7): Z is (1-w0)(1-w4) = (1-w1)(1-w7) = 1/2 descended.\nG3 bridge (section 125 PULL). Multiplication by w on R^2 has det |w|^2 (s: 1/2). Its inverse 1-i is an expanding\n   endomorphism of C/Z[i] of degree N(1-i) = 2, |Z[i]/(1-i)^k| = 2^k, eigenvalues 1+-i: entropy log 2 = 1 bit/tick.\nG4 chain lemmas (e <= 12). Fermat orbit at zeta-order 2^e is one Galois orbit; ord_(2^e)(3) = ord_(2^e)(5) = 2^(e-2);\n   known prime factors of F_k (k = 2..6) are = 1 mod 2^(k+2).\nStage 2 (sealed separately after the log side runs): Z(20), Z(28) for p=2 and Z(12) for p=3 predicted from G2,\n   then measured by lane PSLQ.\n"),
                     ("core_stage2", "77a752092108de26220df059ebd5778a5029937d904acaa1da4215996a0be89b", "NEXUS ask-the-field stage 2 - sealed before any lane value is computed, 17 Oct 2026.\nLaw (route back, owner named on n = 8,12,16,24,32): zero routes = Galois core R0 of the Q-relations among the\nreturns' logs L_m = -log(1 - zeta_n^m/sqrt p). Predicted from the returns alone (core_predictions.txt):\n  p=2 n=20 Z=0 | p=2 n=28 Z=0 | p=2 n=36 Z=1 | p=3 n=12 Z=2 | p=3 n=24 Z=2 | p=5 n=8 Z=0 | p=5 n=12 Z=0\nMembership: p=3 n=12, 24: pi*sqrt3 in the lane span, pi not (phase return zeta_12^-1 pairs with sqrt(-3) coefficients);\n  p=2 n=20, 28: pi not in the lane span (no zeta_8 phase return).\nWay home: lane PSLQ relation dimension, Lam_j = sum_k p^(-k n/2)/(n k + j), |coeff| <= 1e12, residual < 10^(-3dps/4).\n")]:
    check(f"seal {name} re-hashes to {h[:8]}...", hashlib.sha256(txt.encode()).hexdigest() == h)
print(f"\n{sum(checks.values())}/{len(checks)} checks pass;", "ALL PASS" if all(checks.values()) else "FAILURES: " + str([k for k, v in checks.items() if not v]))

PASS seal ask_field re-hashes to f97de11f...
PASS seal core_stage2 re-hashes to 77a75209...

47/47 checks pass; ALL PASS
